# Build the blinded development error-audit pack

This notebook reconstructs the locked 64-case, development-only error audit from the completed larger-development evidence. It downloads only the corresponding development images, creates two blinded reviewer packs, and keeps the unblinding key in a separate administrator pack. This is post-hoc hypothesis generation: it cannot change the completed non-promotion result and must not access the official test partition.

In [ ]:
# 1. Mount Drive and set the exact new commit after this notebook is committed and pushed.
from google.colab import drive
from pathlib import Path
import re

drive.mount("/content/drive")
REPOSITORY_COMMIT = "REPLACE_WITH_FULL_PUSHED_COMMIT_SHA"
EVIDENCE_COMMIT = "7ad3807915d0e461ed77d437b6df436eab677992"
if not re.fullmatch(r"[0-9a-f]{40}", REPOSITORY_COMMIT):
    raise ValueError("Set REPOSITORY_COMMIT to the full pushed commit SHA")
DRIVE_ROOT = Path("/content/drive/MyDrive/gi_vqa_study1")
EVIDENCE_DIR = DRIVE_ROOT / f"larger-development-{EVIDENCE_COMMIT}"
AUDIT_DIR = DRIVE_ROOT / "development-error-audit-v1"
if not EVIDENCE_DIR.is_dir():
    raise FileNotFoundError(f"Completed evidence directory not found: {EVIDENCE_DIR}")
print("Evidence:", EVIDENCE_DIR)
print("Audit output:", AUDIT_DIR)

In [ ]:
# 2. Create a fresh exact checkout.
import shutil
import subprocess

REPOSITORY_URL = "https://github.com/dizza01/VLM.git"
REPOSITORY_ROOT = Path("/content/VLM-development-error-audit")
PROJECT_ROOT = REPOSITORY_ROOT / "gi_vqa_research"
if REPOSITORY_ROOT.exists():
    shutil.rmtree(REPOSITORY_ROOT)
subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_ROOT)], check=True)
subprocess.run(["git", "checkout", "--detach", REPOSITORY_COMMIT], cwd=REPOSITORY_ROOT, check=True)
observed = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_ROOT, check=True,
    capture_output=True, text=True,
).stdout.strip()
status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=REPOSITORY_ROOT, check=True,
    capture_output=True, text=True,
).stdout
if observed != REPOSITORY_COMMIT or status:
    raise RuntimeError("Exact clean checkout verification failed")
print("Checkout PASS:", observed)
%cd /content/VLM-development-error-audit/gi_vqa_research

In [ ]:
# 3. Install data dependencies and authenticate to Hugging Face.
import os
import sys
from google.colab import userdata
from huggingface_hub import login

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "-e", ".[data]", "fsspec==2024.12.0"],
    cwd=PROJECT_ROOT, check=True,
)
hf_token = userdata.get("HF_TOKEN")
if not isinstance(hf_token, str) or not hf_token.strip():
    raise RuntimeError("Add HF_TOKEN in Colab Secrets and enable notebook access")
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
del hf_token
print("Dependencies and authentication PASS")

In [ ]:
# 4. Reconstruct the ignored development artifact required for exact questions.
environment = os.environ.copy()
environment["PYTHONPATH"] = "src"
completed = subprocess.run(
    [
        sys.executable, "-m", "gi_vqa.cli", "materialize-splits",
        "--manifest", "protocols/study1/grouped_split_manifest.json",
        "--project-root", ".",
    ],
    cwd=PROJECT_ROOT, env=environment, text=True, capture_output=True,
)
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
    raise RuntimeError("Development split materialization failed")
print("Development artifact PASS")

In [ ]:
# 5. Build the deterministic blinded pack and materialize its 64 development images.
command = [
    sys.executable, "-m", "gi_vqa.development_error_audit",
    "--project-root", ".",
    "--evidence-dir", str(EVIDENCE_DIR),
    "--output-dir", str(AUDIT_DIR),
    "--specification", "protocols/study1/development_error_audit_v1.json",
    "--materialize-images",
]
subprocess.run(command, cwd=PROJECT_ROOT, env=environment, check=True)
print("Blinded development error-audit pack PASS")

In [ ]:
# 6. Verify the test seal, blinding and image count.
import csv
import json

manifest = json.loads((AUDIT_DIR / "audit_manifest.json").read_text(encoding="utf-8"))
if manifest.get("status") != "PASS" or manifest.get("test_partition_accessed") is not False:
    raise RuntimeError("Audit manifest or test-set seal failed")
if manifest["images"]["materialized_count"] != 64:
    raise RuntimeError("The audit must contain exactly 64 correct development images")
with (AUDIT_DIR / "reviewer_1.csv").open(encoding="utf-8") as handle:
    fields = next(csv.DictReader(handle)).keys()
if "condition" in fields or "sequence_confidence" in fields:
    raise RuntimeError("Reviewer sheet contains unblinded fields")
print(json.dumps({
    "status": manifest["status"],
    "items": manifest["selection"]["items"],
    "reason_counts": manifest["selection"]["reason_counts"],
    "images": manifest["images"]["materialized_count"],
    "test_partition_accessed": manifest["test_partition_accessed"],
}, indent=2))

In [ ]:
# 7. Create separate reviewer and administrator archives.
import zipfile

def make_archive(destination, files, include_images=False):
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in files:
            archive.write(path, path.name)
        if include_images:
            for image in sorted((AUDIT_DIR / "images").glob("*.jpg")):
                archive.write(image, f"images/{image.name}")

reviewer_1_archive = DRIVE_ROOT / "development-error-audit-reviewer-1.zip"
reviewer_2_archive = DRIVE_ROOT / "development-error-audit-reviewer-2.zip"
admin_archive = DRIVE_ROOT / "development-error-audit-admin.zip"
make_archive(reviewer_1_archive, [AUDIT_DIR / "README.md", AUDIT_DIR / "reviewer_1.csv"], True)
make_archive(reviewer_2_archive, [AUDIT_DIR / "README.md", AUDIT_DIR / "reviewer_2.csv"], True)
make_archive(admin_archive, [
    AUDIT_DIR / "audit_manifest.json", AUDIT_DIR / "selected_items.json",
    AUDIT_DIR / "unblinding_key.csv", AUDIT_DIR / "adjudication.csv",
])
print("Reviewer 1:", reviewer_1_archive)
print("Reviewer 2:", reviewer_2_archive)
print("Administrator only:", admin_archive)
print("Do not send the administrator archive to reviewers before review is frozen.")